# 🌊 Liquid Foundation Models (LFM)
## Versión alumno · Aprende cómo funcionan las arquitecturas líquidas

En este notebook vas a:

1. 📖 **Entender** por qué no toda la modelización secuencial necesita atención global
2. 🔍 **Explorar** cómo funcionan las convoluciones locales sobre texto
3. 🚪 **Descubrir** qué son los gates y por qué regulan la información
4. ⚖️ **Comparar** tres arquitecturas: Transformer, Liquid e Híbrida
5. ✏️ **Experimentar** modificando parámetros y observando resultados

---

### 💡 Idea clave

Los **Transformers** usan atención global: cada token puede "mirar" a todos los demás.

Los **Liquid Foundation Models** usan una estrategia diferente:
- **Operaciones locales y baratas** (convoluciones) para la mayor parte del trabajo
- **Atención solo en puntos estratégicos** para capturar dependencias lejanas

> Piensa en ello como la diferencia entre:
> - 📞 **Teléfono comunitario**: todos hablan con todos (atención global)
> - 💬 **Conversaciones locales**: cada persona habla con quienes tiene cerca, y de vez en cuando alguien retransmite información a otros grupos (híbrido)

## 1. Preparación

Importamos las herramientas que necesitamos.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import requests
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Verificar dispositivo disponible
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"📱 Usando dispositivo: {device}")

## 2. El dataset: Don Quijote

Usamos **El Quijote** de Cervantes con tokenización a nivel de **carácter**.

¿Por qué caracteres y no palabras?
- ✅ Más simple: no necesitamos un tokenizador complejo
- ✅ Tamaño reducido: el vocabulario es pequeño (~100 caracteres)
- ✅ El modelo aprende patrones a nivel de letras y fragmentos cortos

**Tu primer dato interesante:** El Quijote tiene más de **1 millón de caracteres**.

In [ ]:
# ======================================
# 2.1 Descargar El Quijote
# ======================================

data_path = Path("el_quijote.txt")

if not data_path.exists():
    print("📥 Descargando Don Quijote...")
    url = "https://www.gutenberg.org/files/2000/2000-0.txt"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text = response.text
    
    # Extraer solo el texto del libro (eliminar metadatos de Project Gutenberg)
    start_marker = "*** START OF THE PROJECT GUTENBERG"
    end_marker = "*** END OF THE PROJECT GUTENBERG"
    start_idx = text.find(start_marker) + len(start_marker)
    end_idx = text.find(end_marker)
    text = text[start_idx:end_idx].strip()
    
    data_path.write_text(text)
    print("✅ Descarga completada")

text = data_path.read_text()
print(f"📚 El Quijote tiene {len(text):,} caracteres")

# ======================================
# 2.2 Crear vocabulario (char-level)
# ======================================
# Paso 1: Obtener todos los caracteres únicos
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"🔤 Vocabulario: {vocab_size} caracteres únicos")

# Paso 2: Crear diccionarios para convertir
# stoi: string TO index (carácter → número)
# itos: index TO string (número → carácter)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# Paso 3: Convertir texto a tensor de números
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)
print(f"📊 Tensor de datos: forma {data.shape}, dtype {data.dtype}")

### 🧪 EJERCICIO 1: Explora el vocabulario

Antes de continuar, explora qué caracteres hay en el vocabulario:
- ¿Cuántos son letras?
- ¿Hay caracteres especiales (espacios, puntuación)?
- ¿Hay mayúsculas y minúsculas?

In [ ]:
# Explora el vocabulario
print("🔍 Primeros 20 caracteres del vocabulario:")
print(chars[:20])
print("\n🔍 Últimos 20 caracteres del vocabulario:")
print(chars[-20:])

# Cuenta tipos de caracteres
letters = sum(1 for c in chars if c.isalpha())
digits = sum(1 for c in chars if c.isdigit())
spaces = sum(1 for c in chars if c.isspace())
punct = sum(1 for c in chars if c in '.,;:!?-—()"\'«»¡¿...')

print(f"\n📊 Resumen del vocabulario:")
print(f"   Letras: {letters}")
print(f"   Dígitos: {digits}")
print(f"   Espacios/tabs: {spaces}")
print(f"   Puntuación: {punct}")
print(f"   Otros: {vocab_size - letters - digits - spaces - punct}")

### Función para obtener batches

Esta función nos permite obtener fragmentos de texto para entrenar.

📝 **block_size** = cuántos caracteres usamos como "historial" para predecir el siguiente
📝 **batch_size** = cuántas secuencias procesamos a la vez (en paralelo)

In [ ]:
def get_batch(batch_size=64, block_size=128, device='cpu'):
    """
    Obtiene un batch aleatorio del dataset.
    
    Args:
        batch_size: número de secuencias en el batch
        block_size: longitud de cada secuencia
        device: donde guardar los tensores
    
    Returns:
        x: tensor de entrada (batch_size, block_size)
        y: tensor de destino (batch_size, block_size) - es x desplazado 1 posición
    """
    # Elegir posiciones iniciales aleatorias
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    # Construir secuencias de entrada
    x = torch.stack([data[i:i+block_size] for i in ix])
    
    # Construir secuencias objetivo (lo que queremos predecir)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    
    return x.to(device), y.to(device)

# Ejemplo de uso
x_batch, y_batch = get_batch(batch_size=4, block_size=20, device=device)
print(f"📦 Batch de entrada:  forma {x_batch.shape}")
print(f"📦 Batch de destino:  forma {y_batch.shape}")
print(f"\n🔍 Ejemplo - Primeros 10 caracteres del primer ejemplo:")
print(f"   Entrada:  {''.join(itos[i.item()] for i in x_batch[0][:10])}")
print(f"   Destino:  {''.join(itos[i.item()] for i in y_batch[0][:10])}")

---

## 3. Bloques fundamentales

Ahora llegamos a la parte más interesante. Vamos a ver dos tipos de bloques:

### 3.1 AttentionBlock (como los Transformers)

Este bloque permite que **cada posición "mire" a todas las demás** posiciones de la secuencia.

Componentes:
1. **MultiheadAttention**: la atención propiamente dicha
2. **LayerNorm**: normaliza para estabilizar el entrenamiento
3. **Feed-Forward**: una red pequeña al final

In [ ]:
# ======================================
# 3.1 BLOQUE DE ATENCIÓN (Transformer)
# ======================================

class AttentionBlock(nn.Module):
    """
    Bloque basado en atención multi-cabeza.
    
    Cada token puede "prestar atención" a todos los demás tokens,
    lo que permite capturar dependencias a cualquier distancia.
    
    Pero... ¡es computacionalmente costoso! O(n²) en la longitud de secuencia.
    """
    
    def __init__(self, d_model, n_heads):
        super().__init__()
        # Atención multi-cabeza: múltiples "perspectivas" de atención
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        
        # Normalización de capa
        self.norm = nn.LayerNorm(d_model)
        
        # Red feed-forward al final
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),  # expandir
            nn.GELU(),                         # activación
            nn.Linear(4 * d_model, d_model)   # comprimir
        )
    
    def forward(self, x):
        # Paso 1: Atención causal consigo misma (self-attention)
        # Cada posición solo puede mirar al pasado y a sí misma
        seq_len = x.size(1)
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool),
            diagonal=1
        )
        attn_out, _ = self.attn(x, x, x, attn_mask=causal_mask)
        
        # Paso 2: Normalización + conexión residual
        x = self.norm(x + attn_out)
        
        # Paso 3: Feed-forward con residuo
        x = self.norm(x + self.ff(x))
        
        return x

### 3.2 LiquidBlock (la innovación de los LFMs)

Aquí está la diferencia clave. En lugar de atención global, usamos:

1. **Puerta de entrada (gate1)**: Decide qué información merece ser procesada
2. **Convolución 1D**: Mira solo una ventana local de caracteres
3. **Puerta de salida (gate2)**: Decide qué del resultado merece pasar
4. **Feed-forward**: Una red pequeña al final

📝 **¿Por qué es más eficiente?** Porque la convolución mira una ventana fija (ej: 5 caracteres), no toda la secuencia.

In [ ]:
# ======================================
# 3.2 BLOQUE LIQUID (LA CLAVE DE LOS LFMs)
# ======================================

class LiquidBlock(nn.Module):
    """
    Bloque "líquido" que combina:
    
    1. Puertas (gates): controlan qué información pasa
    2. Convolución local: detecta patrones en ventanas pequeñas
    3. Conexiones residuales: facilita el entrenamiento
    
    Ventaja: O(n * k) donde k es el tamaño del kernel (mucho menor que n²)
    """
    
    def __init__(self, d_model, kernel_size=5):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        
        # Proyección de entrada
        self.in_proj = nn.Linear(d_model, d_model)
        
        # Puerta 1: antes de la convolución
        # Decide qué información ENTRA a la convolución
        self.gate1 = nn.Linear(d_model, d_model)
        
        # Convolución 1D: solo mira k vecinos
        # Usamos mezcla completa entre canales para ganar capacidad expresiva
        self.conv = nn.Conv1d(
            d_model, d_model, 
            kernel_size,
            padding=kernel_size//2  # para mantener el tamaño
        )
        
        # Puerta 2: después de la convolución
        # Decide qué información SALE de la convolución
        self.gate2 = nn.Linear(d_model, d_model)
        
        # Proyección de salida
        self.out_proj = nn.Linear(d_model, d_model)
        
        self.norm2 = nn.LayerNorm(d_model)
        
        # Feed-forward final
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
    
    def forward(self, x):
        # Guardar para la conexión residual
        residual = x
        
        # Normalizar
        x = self.norm1(x)
        
        # ===== GATE 1: Filtrar entrada =====
        # Produce una "máscara" entre 0 y 1
        # Si la señal es importante → puerta abierta (≈1)
        # Si no → puerta cerrada (≈0)
        gate_signal1 = torch.sigmoid(self.gate1(x))
        z = self.in_proj(x) * gate_signal1
        
        # ===== CONVOLUCIÓN: Procesar localmente =====
        # Cambiar formato: (batch, seq, channels) → (batch, channels, seq)
        z = z.transpose(1, 2)
        z = self.conv(z)
        z = z.transpose(1, 2)  # Volver al formato original
        
        # ===== GATE 2: Filtrar salida de convolución =====
        gate_signal2 = torch.sigmoid(self.gate2(x))
        z = z * gate_signal2
        
        # Proyectar a dimensión original
        z = self.out_proj(z)
        
        # ===== Conexión residual =====
        x = residual + z
        
        #antiguo código
        #x = self.norm2(x)
        #x = x + self.ff(x)
        # Nuevo: Normalizar y feed-forward con conexión residual
        x = self.norm2(x + self.ff(x))
        
        return x

### 🤔 ¿Por qué Conv1D?

La **Conv1D** es como una "ventana deslizante" sobre la secuencia:

```
Si kernel_size = 5:

         Ventana deslizante
    ┌─────────────────────┐
    │ i-2 │ i-1 │ i │ i+1 │ i+2 │
    └─────────────────────┘
         ↓
    Para predecir el carácter en posición i,
    el modelo "ve" solo estos 5 caracteres cercanos
```

**Ventajas:**
- ✅ Muy rápido: O(n × k) donde k es el tamaño del kernel
- ✅ Detecta patrones locales: sílabas, prefijos, sufijos
- ✅ "Traduce bien" a texto porque la sintaxis es local

**Limitaciones:**
- ❌ No ve dependencias lejanas directamente
- ❌ Necesita capas stacking para "ampliar" su alcance

In [ ]:
# ======================================
# ILUSTRACIÓN: Cómo trabaja la Conv1D
# ======================================

print("📖 ILUSTRACIÓN DE CONVOLUCIÓN 1D SOBRE TEXTO")
print("=" * 50)
print("\nSupongamos esta frase (caracteres numerados):")
frase = "En un lu-gar"
print(f"   Texto: '{frase}'")

for i, c in enumerate(frase):
    print(f"   Pos {i}: '{c}'")

print(f"\n🎯 kernel_size = 5 significa que para calcular la")
print("   salida en cada posición, el modelo mira 5 caracteres:")
print(f"   - Para 'g' (posición 9): mira 'lu-ga' (pos 6-10)")

print("\n📊 Complejidad comparada:")
print("   - Atención global: O(n²) - todos con todos")
print("   - Conv1D (k=5):   O(n × 5) ≈ O(n) - solo vecinos")

print("\n💡 En una secuencia de 128 caracteres:")
print("   - Atención: 128 × 128 = 16,384 operaciones")
print("   - Conv1D:   128 × 5 = 640 operaciones")
print("   → ¡25 veces más rápido!")

---

## 4. Los tres modelos que vamos a comparar

Ahora construimos tres modelos completos:

### 4.1 MiniTransformer
Solo atención. Todas las capas son `AttentionBlock`. **Referencia de calidad**.

In [ ]:
# ======================================
# 4. MODELOS COMPLETOS
# ======================================

class MiniTransformer(nn.Module):
    """
    Modelo basado纯粹的 en atención (como GPT pequeño).
    
    Capas: Embedding → [AttentionBlock × n] → Norm → Head
    """
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=6, block_size=128):
        super().__init__()
        # Capa de embedding: convierte caracteres → vectores
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(block_size, d_model)
        self.block_size = block_size
        
        # Stack de capas de atención
        self.layers = nn.ModuleList([
            AttentionBlock(d_model, n_heads) for _ in range(n_layers)
        ])
        
        # Normalización final
        self.norm = nn.LayerNorm(d_model)
        
        # Capa de salida: convierte vectores → probabilidades por carácter
        self.head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        # Embedding de token + posición
        B, T = x.shape
        if T > self.block_size:
            raise ValueError(f"Secuencia demasiado larga: {T} > {self.block_size}")
        pos = torch.arange(T, device=x.device)
        x = self.embed(x) + self.pos_embed(pos)[None, :, :]
        
        # Pasar por cada capa
        for layer in self.layers:
            x = layer(x)
        
        # Normalizar y predecir
        x = self.norm(x)
        return self.head(x)

### 4.2 MiniLiquid
Solo bloques Liquid. Todas las capas son `LiquidBlock`. **Referencia de eficiencia**.

In [ ]:
class MiniLiquid(nn.Module):
    """
    Modelo basado puramente en bloques líquidos (sin atención).
    
    Capas: Embedding → [LiquidBlock × n] → Norm → Head
    
    Ventaja: más rápido que Transformer
    Desventaja: puede fallar en dependencias lejanas
    """
    def __init__(self, vocab_size, d_model=128, n_layers=12, kernel_size=9, block_size=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(block_size, d_model)
        self.block_size = block_size
        
        # Más capas liquid compensan la falta de atención
        self.layers = nn.ModuleList([
            LiquidBlock(d_model, kernel_size) for _ in range(n_layers)
        ])
        
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        B, T = x.shape
        if T > self.block_size:
            raise ValueError(f"Secuencia demasiado larga: {T} > {self.block_size}")
        pos = torch.arange(T, device=x.device)
        x = self.embed(x) + self.pos_embed(pos)[None, :, :]
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        return self.head(x)

### 4.3 HybridLiquid (el enfoque LFM real)

Mezcla **inteligente**: bloques Liquid la mayor parte del tiempo, pero con **atención intercalada** cada pocas capas.

La idea: "Usa operaciones baratas donde puedas, y reserva la atención cara para donde sea necesaria."

In [ ]:
class HybridLiquid(nn.Module):
    """
    Modelo híbrido: combina Liquid y Atención.
    
    Estrategia:
    - Cada  capas hay una de atención
    - El resto son bloques Liquid (eficientes)
    
    Este es el enfoque de los Liquid Foundation Models modernos.
    """
    def __init__(self, vocab_size, d_model=128, n_layers=12, kernel_size=9, block_size=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(block_size, d_model)
        self.block_size = block_size
        
        # Construir capas híbridas
        layers = []
        for i in range(n_layers):
            # Una capa de atención cada 2
            if i % 2 == 0:
                layers.append(AttentionBlock(d_model, n_heads=4))
            else:
                layers.append(LiquidBlock(d_model, kernel_size))
        
        self.layers = nn.ModuleList(layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        B, T = x.shape
        if T > self.block_size:
            raise ValueError(f"Secuencia demasiado larga: {T} > {self.block_size}")
        pos = torch.arange(T, device=x.device)
        x = self.embed(x) + self.pos_embed(pos)[None, :, :]
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        return self.head(x)

### 📊 Resumen de los tres modelos

| Modelo | Tipo de capas | Ventaja principal | Limitación |
|--------|---------------|------------------|------------|
| **MiniTransformer** | Solo atención | Captura dependencias largas | Caro computacionalmente |
| **MiniLiquid** | Solo conv+gate | Muy eficiente | Dependencias lejanas |
| **HybridLiquid** | Mixto (1 atención + 2 liquid) | Equilibrio velocidad/calidad | Complejidad de diseño |

---

## 5. Entrenamiento y generación

Ahora viene la parte práctica: **entrenar** los modelos y **generar texto**.

La función de abajo:
1. Entrena el modelo durante varias épocas
2. Guarda el mejor modelo (menor pérdida)
3. Muestra la curva de pérdida
4. Genera texto nuevo con temperatura controlada

In [ ]:
# ======================================
# 5. FUNCIÓN DE ENTRENAMIENTO
# ======================================

def train_and_generate(model, name, epochs=8, batch_size=64, block_size=128, 
                       lr=4e-4, device='cuda', temperature=0.8, save_dir="models"):
    """
    Entrena un modelo y genera texto al final.
    
    Args:
        model: modelo a entrenar
        name: nombre para mostrar
        epochs: número de pasadas por el dataset
        batch_size: ejemplos por batch
        block_size: longitud de secuencia
        lr: learning rate
        device: cpu o cuda
        temperature: 控制 aleatoriedad en generación (0.8 = equilibrado)
        save_dir: carpeta para guardar el mejor modelo
    """
    import os
    os.makedirs(save_dir, exist_ok=True)
    
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    
    print(f"\n{'='*50}")
    print(f"🚀 ENTRENANDO: {name}")
    print(f"{'='*50}")
    
    losses = []
    best_loss = float('inf')
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        steps = 80  # pasos por época (ajusta según tu GPU)
        
        for _ in tqdm(range(steps), desc=f"  Época {epoch+1}/{epochs}"):
            # Obtener batch
            x, y = get_batch(batch_size, block_size, device)
            
            # Forward pass
            logits = model(x)
            
            # Calcular pérdida (cross-entropy)
            loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / steps
        losses.append(avg_loss)
        print(f"  Época {epoch+1:2d} | Loss: {avg_loss:.4f}")
        
        # Guardar si es el mejor
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), f"{save_dir}/{name.replace(' ', '_')}_best.pt")
            print(f"    ⭐ Mejor modelo guardado (loss: {best_loss:.4f})")
    
    # ===== GRÁFICA DE PÉRDIDA =====
    plt.figure(figsize=(10, 4))
    plt.plot(range(1, epochs+1), losses, marker='o', linewidth=2, label=name)
    plt.title(f"📉 Curva de pérdida - {name}", fontsize=14)
    plt.xlabel("Época")
    plt.ylabel("Pérdida (Cross-Entropy)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
    
    # ===== GENERACIÓN DE TEXTO =====
    print(f"\n📝 Generando texto con temperatura = {temperature}")
    print("-" * 60)
    model.eval()
    with torch.no_grad():
        # Empezar con un espacio en blanco
        context = torch.tensor([stoi[' ']], dtype=torch.long, device=device).unsqueeze(0)
        generated = []
        
        for _ in range(400):
            context_window = context[:, -block_size:]
            logits = model(context_window)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            context = torch.cat([context, next_idx], dim=1)
            generated.append(itos[next_idx.item()])
        
        print("\n" + "".join(generated))
        print("\n" + "-" * 60)

### 🤔 ¿Qué es la temperatura?

La **temperatura** controla la **aleatoriedad** de la generación:

| Temperatura | Efecto | Resultado |
|-------------|--------|----------|
| **0.5** | Muy conservadora | Texto repetitivo, seguro |
| **0.8** | Equilibrada | Interesante y coherente |
| **1.0** | Neutral | distribución original |
| **1.2** | Creativa | Texto más loco, a veces incoherente |

💡 **Consejo**: empieza con 0.8 y experimenta desde ahí.

---

## 6. 🏃 EJERCICIO FINAL: Compara los tres modelos

Ahora vas a entrenar los tres modelos y comparar:

1. **¿Cuál converge más rápido?** (pierde menos por época)
2. **¿Cuál genera texto más coherente?**
3. **¿Cuál es más rápido en entrenar?**

**Instrucciones:**
- Ejecuta la celda de abajo
- Observa las gráficas de pérdida
- Lee el texto generado por cada uno
- Responde las preguntas de reflexión al final

In [ ]:
# ======================================
# 6. ENTRENAR LOS TRES MODELOS
# ======================================

if __name__ == "__main__":
    # Configuración
    d_model = 128
    block_size = 128
    save_dir = "saved_models_lfm"
    
    # Parámetros de entrenamiento
    EPOCHS = 8
    TEMPERATURE = 0.85  # 🎚️ ¡Cambia este valor para experimentar!
    
    print(f"📱 Dispositivo: {device}")
    print(f"📊 Vocabulario: {vocab_size} caracteres")
    print(f"⚙️  Config: d_model={d_model}, block_size={block_size}")
    print(f"🌡️  Temperatura: {TEMPERATURE}")
    
    # Definir modelos
    models = [
        ("MiniTransformer", MiniTransformer(vocab_size, d_model, n_heads=4, n_layers=6, block_size=block_size)),
        ("Pure Liquid", MiniLiquid(vocab_size, d_model, n_layers=12, block_size=block_size)),
        ("Hybrid LFM", HybridLiquid(vocab_size, d_model, n_layers=12, block_size=block_size))
    ]
    
    # Entrenar y comparar
    for name, model in models:
        train_and_generate(
            model, 
            name, 
            epochs=EPOCHS, 
            device=device, 
            temperature=TEMPERATURE,
            save_dir=save_dir
        )

---

## 7. 📝 Reflexión: Responde estas preguntas

Después de ver los resultados, piensa en estas cuestiones:

### Pregunta 1: Velocidad de convergencia
¿Cuál modelo alcanza menor pérdida? ¿Era lo esperado?

### Pregunta 2: Texto generado
Observa los textos generados. ¿Cuál suena más "a Don Quijote"?
- ¿Reconoces fragmentos que parezcan del libro?
- ¿Cuál tiene más "inventado"/incorrecto?

### Pregunta 3: Trade-off
Si tuvieras que elegir un modelo para producción:
- ¿Priorizarías calidad (menor pérdida) o velocidad?
- ¿Qué arquitectura elegirías para cada caso?

### Pregunta 4: Experimenta
🔬 **Desafío extra**: Antes de ejecutar, intenta predecir:
1. ¿Qué pasa si cambias `TEMPERATURE` a `0.5`?
2. ¿Y si la cambias a `1.2`?
3. ¿Qué pasaría si usaras `kernel_size=3` en lugar de `5`?

Después ejecuta y verifica tus hipótesis.

---

## 8. 🔬 EXPERIMENTOS OPCIONALES

Una vez que hayas entendido el código base, aquí tienes ideas para experimentar:

### Experimento A: Cambia el tamaño del kernel
```python
# Kernel más pequeño → mayor eficiencia, pero menos contexto local
model = MiniLiquid(vocab_size, d_model=128, n_layers=12, kernel_size=3)
```

### Experimento B: Más o menos atención en el híbrido
```python
# Más atención (cada 2 capas en vez de cada 3)
# Modifica la condición en HybridLiquid: if i % 2 == 0
```

### Experimento C: Modelo más grande
```python
# d_model=256, n_layers=16
model = HybridLiquid(vocab_size, d_model=256, n_layers=16)
```

### Experimento D: Block size diferente
```python
# Block size más corto → secuencias más fáciles, pero menos contexto
# (necesitas cambiar block_size en train_and_generate también)
```

---

## 📚 Resumen de lo que has aprendido

✅ **Atención global (Transformer)**: cada token mira a todos. Potente pero caro.

✅ **Convolución local (Liquid)**: cada token mira solo k vecinos. Eficiente pero limitado.

✅ **Puertas (Gates)**: regulan qué información pasa. Añaden "inteligencia" al modelo.

✅ **Híbrido**: combina lo mejor de ambos mundos. Es el enfoque de los LFMs reales.

---

> 💬 **Frase para recordar**:
> 
> *"Los Transformers nos enseñaron que atender globalmente funciona muy bien. Los LFMs nos recuerdan que no siempre hace falta pagar ese coste en todas las capas"*.

---

🎉 **¡Felicidades! Has completado el notebook de Liquid Foundation Models!**